<span style="color:#888888">Copyright (c) 2014-2021 National Technology and Engineering Solutions of Sandia, LLC. Under the terms of Contract DE-NA0003525 with National Technology and Engineering Solutions of Sandia, LLC, the U.S. Government retains certain rights in this software.     Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met:</span>

<span style="color:#888888">1. Redistributions of source code must retain the above copyright notice, this list of conditions and the following disclaimer.</span>

<span style="color:#888888">2. Redistributions in binary form must reproduce the above copyright notice, this list of conditions and the following disclaimer in the documentation and/or other materials provided with the distribution.</span>

<span style="color:#888888">THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.</span>

# <span style="color:#0054a8">**Tutorial 2:**</span> <span style="color:#555555">How to Create Trajectories from a Deliminated File</span>

In [1]:
import tracktable.examples.tutorials.tutorial_helper as tutorial 

## Purpose

This notebook demonstrates how to create Tracktable trajectorys from a deliminated (e.g. csv, tsv, etc.) data file.  A data file must contain the following columns in order to be compatible with Tracktable:

* **<span style="color:#00add0">an identifier</span>** that is unique to each object
* **<span style="color:#00add0">a timestamp</span>**
* **<span style="color:#00add0">longitude</span>**
* **<span style="color:#00add0">latitude</span>**

Both ordering and headers for these columns can vary, but they must exist in the file.  Each row of the data file should represent the information for a single trajectory point.

**<span style="color:#81062e">IMPORTANT:</span>** Deliminated files must be **sorted by timestamp** to be compatible with Tracktable.

*Note:* If you want to create individual Trajectory Point objects, but not assemble them into Trajectory objects, please see [Tutorial 1](Tutorial_01.ipynb).

## Step 1: Set up a TrajectoryPointReader object.

We will use the provided example data $^1$ for this tutorial.  For the sake of brevity, the function below executes the steps from [Tutorial 1](Tutorial_01.ipynb) to create a TrajectoryPointReader object.

In [2]:
reader = tutorial.create_point_reader()

## Step 2: Create an AssembleTrajectoryFromPoints object.

This will build trajectories from the individual points.

In [3]:
from tracktable.applications.assemble_trajectories import AssembleTrajectoryFromPoints

builder = AssembleTrajectoryFromPoints()

We must tell the builder to get the trajectory points from our reader.

In [4]:
builder.input = reader

### *Additional Settings*

How far apart (in km) should sequential points (with the same object ID) have to be before we consider them separate trajectories?  This parameter is optional and defaults to `None`.

In [5]:
builder.separation_distance = 10 # km

How far apart (in time) should sequential points (with the same object ID) have to be before we consider them separate trajectories?  This parameter is optional and defaults to 30 minutes.

In [6]:
from datetime import timedelta

builder.separation_time = timedelta(minutes = 20)

What is the minimum number of points that a trajectory must have?  Any trajectories assembled with fewer than this number will be discarded.  This parameter is optional and defaults to 2 points.

In [7]:
builder.minimum_length = 5 # points

## Step 3: Assemble Trajectories from Point Data

In [8]:
trajectories = list(builder)

INFO:tracktable.applications.assemble_trajectoriesAssembleTrajectoryFromPoints:New trajectories will be declared after a separation of 10 distance units between two points or a time lapse of at least 0:20:00 (hours, minutes, seconds).
INFO:tracktable.applications.assemble_trajectoriesAssembleTrajectoryFromPoints:Trajectories with fewer than 5 points will be discarded.
INFO:tracktable.applications.assemble_trajectoriesAssembleTrajectoryFromPoints:Done assembling trajectories. 279 trajectories produced and 22 discarded for having fewer than 5 points.


How many trajectories do we have?

In [9]:
len(trajectories)

279

## Step 4: Accessing Trajectory Information

For each `trajectory`, we can access the following information:

* `trajectory.object_id`: a string identifier that is unique to each object
* `trajectory.trajectory_id`: a string identifier that is mostly-unique to each trajectory, created by concatenating the object ID, start timestamp and end timestamp together

This is demonstrated below for our first ten trajectories.

In [10]:
for trajectory in trajectories[:10]:
    object_id      = trajectory.object_id
    trajectory_id  = trajectory.trajectory_id
    
    print(f'Object ID: {object_id}')
    print(f'Trajectory ID: {trajectory_id}\n')

Object ID: 367109000
Trajectory ID: 367109000_20200630000104_20200630002505

Object ID: 367484710
Trajectory ID: 367484710_20200630000243_20200630002642

Object ID: 367000140
Trajectory ID: 367000140_20200630000000_20200630005959

Object ID: 366999618
Trajectory ID: 366999618_20200630000000_20200630005949

Object ID: 367776270
Trajectory ID: 367776270_20200630000000_20200630005952

Object ID: 367022550
Trajectory ID: 367022550_20200630000000_20200630005919

Object ID: 367515850
Trajectory ID: 367515850_20200630000000_20200630005941

Object ID: 367531640
Trajectory ID: 367531640_20200630000000_20200630005950

Object ID: 338531000
Trajectory ID: 338531000_20200630000000_20200630005955

Object ID: 366516370
Trajectory ID: 366516370_20200630000000_20200630005940



## Step 5: Accessing Trajectory Point Information

Let's look at just the first trajectory in our list:

In [11]:
trajectory = trajectories[0]

Trajectory points can be accessed in a trajectory object using list indexing.  So, we can get the first point in our trajectory as follows:

In [12]:
trajectory_point = trajectory[0]

The information from the required columns of the csv can be accessed for a single `trajectory_point` object as

* **<span style="color:#00add0">unique object identifier:</span>** `trajectory_point.object_id`
* **<span style="color:#00add0">timestamp:</span>** `trajectory_point.timestamp`
* **<span style="color:#00add0">longitude:</span>** `trajectory_point[0]`
* **<span style="color:#00add0">latitude:</span>** `trajectory_point[1]`

The optional column information is available through the member variable `properties` as follows: `trajectory_point.properties['what-you-named-it']`.

Below, we access all of the information stored in our `trajectory_point` object.

In [13]:
object_id    = trajectory_point.object_id
timestamp    = trajectory_point.timestamp
longitude    = trajectory_point[0]
latitude     = trajectory_point[1]
heading      = trajectory_point.properties["heading"]
vessel_name  = trajectory_point.properties["vessel-name"]
eta          = trajectory_point.properties["eta"]

print(f'Unique ID: {object_id}')
print(f'Timestamp: {timestamp}')
print(f'Longitude: {longitude}')
print(f'Latitude: {latitude}')
print(f'Heading: {heading}')
print(f'Vessel Name: {vessel_name}')
print(f'ETA: {eta}\n')

Unique ID: 367109000
Timestamp: 2020-06-30 00:01:04+00:00
Longitude: -74.2053
Latitude: 40.60922
Heading: 189.0
Vessel Name: OVERSEAS HOUSTON
ETA: 2020-06-30 13:40:04+00:00



## What can we do with trajectories in Tracktable?

* Compact trajectory storage using `.traj` files is discussed in Tutorials [3](Tutorial_03.ipynb) & [4](Tutorial_04.ipynb).
* Tutorials [5A](Tutorial_05A.ipynb) & [5B](Tutorial_05B.ipynb) demonstrate how to visualize trajectories with Tracktable.
* Filtering trajectories using the capabilities provided by the `geomath` module is shown in [Tutorial 6](Tutorial_06.ipynb).

<span style="color:gray">$^1$ Bureau of Ocean Energy Management (BOEM) and National Oceanic and Atmospheric Administration (NOAA). MarineCadastre.gov. *AIS Data for 2020.* Retrieved February 2021 from [marinecadastre.gov/data](https://marinecadastre.gov/data/).  Trimmed down to the first hour of June 30, 2020, restricted to in NY Harbor.</span>